# Object Recognition Based on YOLO (Deep Neural Network)

This section introduces how to implement common object recognition using YOLO (Deep Neural Network) + OpenCV.

## Preparation
Since the product automatically runs the main program on startup, and the main program occupies the camera resources, this tutorial cannot be used in that state. You need to stop the main program or disable its automatic startup before restarting the robot.

It should be noted that because the robot's main program uses multithreading and is configured to start automatically via a service, the usual method of `sudo killall python` often does not work. Therefore, we will introduce a method to disable the main program's automatic startup.

If you have already disabled the automatic startup of the robot's main program, you do not need to perform the following "Stopping the Main Program" section.

### Stopping the Main Program
1. Click the "" icon next to the tab at the top of this page to open a new tab named Launcher.
2. Click Terminal under Other to open the terminal window.
3. In the terminal window, type `bash` and press Enter.
4. You can now use the Bash Shell to control the robot.
5. Enter the command: `systemctl --user stop ugv-app.service`.
6. On the terminal page, after the command has been executed, continue with the remaining part of this tutorial.

## Example

The following code block can be run directly:

1. Select the code block below.
2. Press Shift + Enter to run the code block.
3. Watch the real-time video window.
4. Press STOP to close the real-time video and release the camera resources.

### If you cannot see the real-time camera feed when running:

- Click on Kernel -> Shut down all kernels above.
- Close the current section tab and open it again.
- Click `STOP` to release the camera resources, then run the code block again.
- Reboot the device.

### Features of This Section

The `yolov8n.engine` file are in the same path as this .ipynb file.

When the code blocks run successfully, you can point the camera at common objects such as: 

class_names = ['person','bicycle','car','motorcycle','airplane','bus','train','truck','boat','traffic light',
                'fire hydrant','stop sign','parking meter','bench','bird','cat','dog','horse','sheep','cow',
                'elephant','bear','zebra','giraffe','backpack','umbrella','handbag','tie','suitcase','frisbee',
                'skis','snowboard','sports ball','kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket',
                'bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple','sandwich',
                'orange','broccoli','carrot','hot dog','pizza','donut','cake','chair','couch','potted plant',
                'bed','dining table','toilet','tv','laptop','clock','vase','scissors','teddy bear','hair drier','toothbrush'].

The program will annotate the objects it recognizes in the image and label them with their names.

In [ ]:
import cv2 # Import the OpenCV library for image processing
import numpy as np # A library for mathematical calculations
from IPython.display import display, Image # Used to display images in Jupyter Notebook
import ipywidgets as widgets # Used to create widgets for interactive interfaces, such as buttons
import threading # Used to create new threads for asynchronous task execution
from ultralytics import YOLO

# Predefined category names, set according to the Caffe model.
net = YOLO('yolov8n.engine')
class_names = ['person','bicycle','car','motorcycle','airplane','bus','train','truck','boat','traffic light',
                'fire hydrant','stop sign','parking meter','bench','bird','cat','dog','horse','sheep','cow',
                'elephant','bear','zebra','giraffe','backpack','umbrella','handbag','tie','suitcase','frisbee',
                'skis','snowboard','sports ball','kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket',
                'bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple','sandwich',
                'orange','broccoli','carrot','hot dog','pizza','donut','cake','chair','couch','potted plant',
                'bed','dining table','toilet','tv','laptop','clock','vase','scissors','teddy bear','hair drier','toothbrush']

# Create a "Stop" button that users can click to stop the video stream.
# ================
stopButton = widgets.ToggleButton(
    value=False,
    description='Stop',
    disabled=False,
    button_style='danger', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Description',
    icon='square' # (FontAwesome names without the `fa-` prefix)
)


# Define display functions to process video frames and perform object detection.
# ================
def view(button):
    camera = cv2.VideoCapture(0) # Create a camera instance
    if not camera.isOpened():
        print("Error: Camera not accessible.")
        return
    #Set resolution
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    
    display_handle=display(None, display_id=True)  # Create a display handle for updating the displayed image
    i = 0
    
    avg = None
    
    while True:
        _, frame = camera.read() # Capture a frame from the camera
        # frame = cv2.flip(frame, 1) # if your camera reverses your image
        if frame is None:
            break

        height, width = frame.shape[:2]  # Get the height and width of the image

        results = net(frame, verbose=False)
        r = results[0]

        if r.boxes is not None:

            boxes = r.boxes.xyxy.cpu().numpy()
            scores = r.boxes.conf.cpu().numpy()
            classes = r.boxes.cls.cpu().numpy()

            for box, score, cls in zip(boxes, scores, classes):
                if score > 0.2:
                    x1, y1, x2, y2 = map(int, box)

                    label = "{}: {:.2f}%".format(class_names[int(cls)], score * 100)

                    cv2.rectangle(frame,(x1, y1), (x2, y2),(0, 255, 0), 1)
                    cv2.putText(frame,label,(x1, max(20, y1 - 10)),cv2.FONT_HERSHEY_SIMPLEX,0.7, (0, 255, 0), 1)

        _, frame = cv2.imencode('.jpeg', frame)  # Encode the frame as JPEG format
        display_handle.update(Image(data=frame.tobytes()))  # Update the displayed image
        if stopButton.value==True:  # Check if the "Stop" button is pressed.
            camera.release() # If so, turn off the camera.
            display_handle.update(None)  # Clear the displayed content

            
# Display the "Stop" button and start the thread for the display function.
# ================
display(stopButton)
thread = threading.Thread(target=view, args=(stopButton,))
thread.start()